# Análisis Exploratorio de Datos (EDA) - Marketing Bancario

El propósito de este notebook es realizar un análisis exploratorio de los datos de marketing bancario para entender mejor las características del conjunto de datos, identificar patrones, anomalías, y obtener información relevante que pueda guiar el preprocesamiento de datos y la construcción del modelo de predicción.

## Cargar Bibliotecas y Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración para visualizaciones
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Cargar el conjunto de datos
# Asumimos que el archivo está en la ruta especificada y separado por punto y coma.
# Para la ejecución real, nos aseguraríamos de que el archivo exista.
try:
    df = pd.read_csv('../data/bank-full.csv', sep=';')
except FileNotFoundError:
    print("Error: El archivo 'bank-full.csv' no se encontró en '../data/'.")
    print("Por favor, asegúrese de que el archivo de datos esté en la ubicación correcta.")
    # En un entorno real, podríamos intentar descargarlo o detener la ejecución.
    # Para este ejercicio, crearemos un DataFrame vacío si no se encuentra.
    df = pd.DataFrame() 

## Inspección Inicial de los Datos

Realizaremos una inspección inicial para entender la estructura, tipos de datos y la presencia de valores nulos o desconocidos.

In [ ]:
if not df.empty:
    print("Primeras 5 filas del DataFrame:")
    display(df.head())

In [ ]:
if not df.empty:
    print("\nInformación general del DataFrame:")
    df.info()

In [ ]:
if not df.empty:
    print("\nDimensiones del DataFrame (filas, columnas):")
    print(df.shape)

In [ ]:
if not df.empty:
    print("\nTipos de datos de cada columna:")
    print(df.dtypes)

In [ ]:
if not df.empty:
    print("\nConteo de valores nulos estándar por columna:")
    print(df.isnull().sum())

In [ ]:
if not df.empty:
    print("\nConteo de valores 'unknown' en columnas seleccionadas:")
    cols_with_unknown = ['job', 'education', 'marital', 'housing', 'loan', 'poutcome'] # 'contact' también podría tener 'unknown'
    if 'contact' in df.columns:
        cols_with_unknown.append('contact')
        
    for col in cols_with_unknown:
        if col in df.columns:
            unknown_count = df[col].isin(['unknown']).sum()
            if unknown_count > 0:
                 print(f"Columna '{col}': {unknown_count} valores 'unknown'")
        else:
            print(f"Advertencia: La columna '{col}' no se encontró en el DataFrame.")

## Análisis de la Variable Objetivo (`y`)

La variable objetivo `y` indica si el cliente suscribió un depósito a plazo. Es crucial entender su distribución.

In [ ]:
if not df.empty and 'y' in df.columns:
    plt.figure(figsize=(6,4))
    sns.countplot(x='y', data=df)
    plt.title('Distribución de la Variable Objetivo (y)')
    plt.xlabel('Suscribió Depósito a Plazo')
    plt.ylabel('Cantidad')
    plt.show()
else:
    print("La columna 'y' no está presente o el DataFrame está vacío.")

In [ ]:
if not df.empty and 'y' in df.columns:
    print("\nPorcentaje de cada clase en la variable objetivo 'y':")
    print(df['y'].value_counts(normalize=True) * 100)
else:
    print("La columna 'y' no está presente o el DataFrame está vacío.")

Se observa un desbalance de clases, con una proporción mayor de 'no' que de 'yes'. Esto deberá ser considerado durante el modelado (e.g., usando técnicas de remuestreo o métricas de evaluación apropiadas).

## Análisis de Variables Numéricas

Exploraremos la distribución, tendencia central, dispersión y presencia de outliers en las variables numéricas.

In [ ]:
if not df.empty:
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
    if numerical_cols:
        print("Estadísticas descriptivas de las variables numéricas:")
        display(df[numerical_cols].describe())
    else:
        print("No se encontraron columnas numéricas.")
else:
    print("El DataFrame está vacío.")

In [ ]:
if not df.empty and numerical_cols:
    for col in numerical_cols:
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        sns.histplot(df[col], kde=True)
        plt.title(f'Histograma de {col}')
        
        plt.subplot(1, 2, 2)
        sns.boxplot(y=df[col])
        plt.title(f'Boxplot de {col}')
        
        plt.tight_layout()
        plt.show()
elif not df.empty:
    print("No hay columnas numéricas para graficar.")

**Observaciones sobre Variables Numéricas:**
*   **age**: Distribución relativamente normal, con algunos outliers en edades mayores.
*   **balance**: Altamente sesgada a la derecha, con muchos outliers indicando saldos muy altos (y también negativos, lo que podría ser deuda o un error de dato).
*   **day**: Distribución uniforme, como es de esperar para los días del mes.
*   **duration**: Muy sesgada a la derecha. La duración de la llamada es un factor importante (como se indica en la descripción del dataset, este atributo afecta mucho al resultado; si duration=0, entonces y='no'. Sin embargo, no se conoce antes de realizar la llamada y debería descartarse para un modelo predictivo realista si el objetivo es predecir *antes* de llamar. Para este EDA, lo mantenemos).
*   **campaign**: Mayoría de valores bajos, indicando pocos contactos en la campaña actual. Sesgada a la derecha con outliers.
*   **pdays**: Muchos valores en -1 (cliente no contactado previamente). Para los contactados, la distribución está sesgada.
*   **previous**: Similar a 'campaign', la mayoría de los clientes tienen pocos o ningún contacto previo. Sesgada a la derecha.

## Análisis de Variables Categóricas

Examinaremos la frecuencia de cada categoría en las variables cualitativas.

In [ ]:
if not df.empty:
    categorical_cols = df.select_dtypes(include='object').columns.tolist()
    # Excluimos 'y' si ya fue analizada o la incluimos si es de tipo objeto y no fue separada antes
    if 'y' in categorical_cols:
        categorical_cols.remove('y') # Ya se analizó como variable objetivo
    
    print(f"Columnas categóricas identificadas: {categorical_cols}")
else:
    print("El DataFrame está vacío.")

In [ ]:
if not df.empty and categorical_cols:
    for col in categorical_cols:
        plt.figure(figsize=(10, 6))
        df[col].value_counts().plot(kind='bar')
        plt.title(f'Distribución de {col}')
        plt.ylabel('Cantidad')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        
        print(f"\nValue counts para {col}:")
        print(df[col].value_counts(dropna=False))
        print("-------------------------------------")
elif not df.empty:
    print("No hay columnas categóricas para graficar.")

**Observaciones sobre Variables Categóricas:**
*   **job**: La categoría 'blue-collar', 'management', y 'technician' son las más comunes. Hay valores 'unknown'.
*   **marital**: 'Married' es la categoría predominante.
*   **education**: 'Secondary' es el nivel educativo más frecuente, seguido por 'tertiary'. Hay valores 'unknown'.
*   **default**: La gran mayoría no tiene crédito en default ('no').
*   **housing**: Una ligera mayoría tiene préstamo hipotecario ('yes').
*   **loan**: La gran mayoría no tiene préstamo personal ('no').
*   **contact**: El tipo de contacto 'cellular' es el más común. Hay valores 'unknown'.
*   **month**: Los meses con más contactos son 'may', 'jul', 'aug'.
*   **poutcome**: 'unknown' es la categoría más frecuente para el resultado de la campaña anterior, indicando que para muchos clientes no se conoce este dato. 'failure' y 'success' son menos comunes.

## Análisis de Correlación (entre variables numéricas)

Calcularemos la matriz de correlación para identificar relaciones lineales entre las variables numéricas.

In [ ]:
if not df.empty and numerical_cols:
    correlation_matrix = df[numerical_cols].corr()
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Heatmap de Correlación de Variables Numéricas')
    plt.show()
else:
    print("No hay suficientes columnas numéricas para calcular la correlación o el DataFrame está vacío.")

**Discusión de Correlaciones Significativas:**
*   Se observa una correlación positiva moderada entre `pdays` y `previous` cuando `pdays` no es -1 (lo cual tiene sentido, ya que `pdays` es el número de días desde el último contacto y `previous` el número de contactos). Sin embargo, la forma en que se codifica `pdays` (-1 para no contactado) complica la interpretación directa de la correlación lineal general.
*   La mayoría de las otras variables numéricas no muestran correlaciones lineales fuertes entre sí.
*   `duration` podría tener correlación con la variable objetivo, pero como se mencionó, su uso como predictor es problemático si se busca predecir *antes* de la llamada.

## Conclusiones del EDA

Este análisis exploratorio inicial nos ha proporcionado varias ideas clave:

1.  **Calidad de los Datos:**
    *   No hay valores nulos estándar (NaN), pero sí existen valores 'unknown' en varias columnas categóricas (`job`, `education`, `contact`, `poutcome`). Estos deberán ser tratados (imputación, considerar como categoría separada, etc.).
    *   La columna `balance` tiene valores negativos, lo que podría ser interpretado como deuda o requerir clarificación.

2.  **Variable Objetivo (`y`):
    *   Existe un desbalance de clases significativo: aproximadamente 88% 'no' y 12% 'yes'. Esto es muy importante para la fase de modelado, ya que podría sesgar el modelo hacia la clase mayoritaria. Se necesitarán técnicas como remuestreo (oversampling de la clase minoritaria, undersampling de la mayoritaria), uso de pesos de clase en el modelo, o selección de métricas de evaluación adecuadas (e.g., F1-score, AUC-PR) en lugar de solo accuracy.

3.  **Variables Numéricas:**
    *   Varias variables numéricas (`balance`, `duration`, `campaign`, `pdays`, `previous`) presentan distribuciones sesgadas y outliers. Esto sugiere que podrían beneficiarse de transformaciones (e.g., logarítmica) y que el tratamiento de outliers podría ser necesario para algunos modelos sensibles a ellos.
    *   `duration`: Es un predictor fuerte de la variable objetivo, pero su uso es problemático para predecir si un cliente *suscribirá* antes de la llamada. Si el objetivo es predecir el resultado de una llamada ya hecha, puede usarse; si es predecir *antes* de llamar, debe excluirse o manejarse con mucho cuidado.
    *   `pdays`: El valor -1 tiene un significado especial (no contactado previamente). Esto podría requerir una transformación o creación de una variable binaria para indicar si fue contactado previamente.

4.  **Variables Categóricas:**
    *   Algunas categorías tienen muy pocos miembros, lo que podría causar problemas en la codificación (e.g., one-hot encoding). Se podría considerar agrupar categorías minoritarias.
    *   La alta presencia de 'unknown' en `poutcome` sugiere que la información de campañas anteriores es escasa para una gran parte de los clientes.

5.  **Correlaciones:**
    *   No se observaron correlaciones lineales muy fuertes entre las variables predictoras numéricas, lo que reduce problemas de multicolinealidad directa entre ellas.

**Próximos Pasos (Implicaciones para Feature Engineering y Modelado):**
*   **Preprocesamiento:**
    *   Manejo de valores 'unknown' (imputación o categoría separada).
    *   Codificación de variables categóricas (e.g., one-hot encoding, target encoding).
    *   Transformación de variables numéricas sesgadas.
    *   Escalado de características numéricas (e.g., StandardScaler, MinMaxScaler) especialmente para modelos sensibles a la escala como SVM o redes neuronales.
    *   Tratamiento de outliers (capping, removal, o uso de modelos robustos).
    *   Creación de nuevas características (e.g., a partir de `pdays` para indicar si fue contactado).
*   **Modelado:**
    *   Abordar el desbalance de clases.
    *   Seleccionar métricas de evaluación apropiadas.
    *   Considerar la exclusión de `duration` si el objetivo es la predicción pre-llamada.
    *   Experimentar con diferentes algoritmos, incluyendo modelos de deep learning que son el objetivo final de este proyecto.